# 🇬🇧 FastSpeech 2 — English TTS (FIXED Notebook)

**All known errors fixed:**
- ✅ HuggingFace gated dataset login
- ✅ Correct Svarah split (`test`) and audio key (`audio_filepath`)
- ✅ MFA installed via Miniconda (not bare conda)
- ✅ Training runs in Colab system Python (torch available)
- ✅ All output directories created before use
- ✅ HiFi-GAN downloaded via gdown

> **Before running:** Set `Runtime → Change runtime type → T4 GPU`

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — fix runtime!')
print('CUDA:', torch.cuda.is_available())
!nvidia-smi | head -20


## Step 1 — Install All Dependencies

**What this does:**
1. Installs system audio tools (`espeak-ng`, `ffmpeg`, `sox`)
2. Installs all Python packages in Colab's system Python (keeps `torch` working)
3. Installs **Miniconda** to `~/miniconda` and uses it **only for MFA**
4. Downloads MFA acoustic model and dictionary

> ⏱️ Takes ~5-10 minutes. Run once per session.

In [ ]:
# --- System packages ---
!apt-get install -y -q espeak-ng espeak-ng-data libespeak-ng-dev sox ffmpeg

# --- Python packages (all in Colab's system Python so torch stays available) ---
!pip install -q --upgrade pip
!pip install -q \
    phonemizer \
    epitran \
    num2words \
    tgt \
    pyworld \
    librosa \
    soundfile \
    pyyaml \
    tqdm \
    indic-transliteration \
    datasets \
    huggingface_hub \
    seaborn \
    gdown \
    pandas \
    matplotlib

print('System + pip packages installed!')

# Verify torch is still working
import torch
print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


In [ ]:
import os, subprocess, sys

CONDA_DIR = os.path.expanduser('~/miniconda')
MFA_BIN   = os.path.join(CONDA_DIR, 'bin', 'mfa')

if os.path.exists(MFA_BIN):
    print('MFA already installed, skipping.')
else:
    print('Installing Miniconda...')
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    !bash /tmp/miniconda.sh -b -u -p {CONDA_DIR}
    print('Miniconda installed!')

    # Add to PATH for this session
    os.environ['PATH'] = CONDA_DIR + '/bin:' + os.environ.get('PATH', '')

    # Accept conda ToS (required since conda 23.x)
    !{CONDA_DIR}/bin/conda config --set always_yes true
    !{CONDA_DIR}/bin/conda config --set channel_priority flexible

    print('Installing MFA via conda (takes ~3-5 min)...')
    !{CONDA_DIR}/bin/conda install -c conda-forge montreal-forced-aligner -y -q
    print('MFA installed!')

# Always add conda bin to PATH
os.environ['PATH'] = CONDA_DIR + '/bin:' + os.environ.get('PATH', '')

# Verify MFA
result = subprocess.run([MFA_BIN, 'version'], capture_output=True, text=True)
if result.returncode == 0:
    print('MFA version:', result.stdout.strip())
else:
    print('MFA check failed:', result.stderr[:200])

# Download acoustic + dictionary models
print('Downloading MFA models...')
!{MFA_BIN} model download acoustic english_us_arpa 2>/dev/null || echo 'Already downloaded or failed'
!{MFA_BIN} model download dictionary english_us_arpa 2>/dev/null || echo 'Already downloaded or failed'
print('MFA setup complete!')


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# ✏️ CHANGE THIS — your GitHub username
GITHUB_USERNAME = 'YOUR_GITHUB_USERNAME'
REPO_NAME       = 'FastSpeech2_Assignment'

if os.path.exists('/content/FastSpeech2/English/train.py'):
    print('Repo already cloned, skipping.')
else:
    clone_url = f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'
    !git clone {clone_url} /content/FastSpeech2

os.chdir('/content/FastSpeech2')
import sys
sys.path.insert(0, '/content/FastSpeech2')

# Create all output dirs up front (prevents FileNotFoundError later)
dirs_to_create = [
    'English/checkpoints',
    'Results/English/test_generated_audios',
    'Results/English/variance_adaptor_plots/duration',
    'Results/English/variance_adaptor_plots/pitch',
    'Results/English/variance_adaptor_plots/energy',
    'data/raw/english/wavs',
    'data/mfa_corpus/english',
    'data/mfa_aligned/english',
    'data/preprocessed/english',
    '/content/drive/MyDrive/FastSpeech2/Results/English',
    '/content/drive/MyDrive/FastSpeech2/English/checkpoints',
]
for d in dirs_to_create:
    os.makedirs(d, exist_ok=True)

print('Working dir:', os.getcwd())
!ls /content/FastSpeech2


In [ ]:
import os
required = [
    'English/preprocess.py', 'English/train.py',
    'English/inference.py',  'English/config.yaml',
    'model/fastspeech2.py',  'model/variance_adaptor.py',
    'model/modules.py',      'model/loss.py',
    'utils/audio.py',        'utils/text_english.py',
    'utils/plot_utils.py',
]
all_ok = True
for f in required:
    full = f'/content/FastSpeech2/{f}'
    ok   = os.path.exists(full)
    print('OK    ' if ok else 'MISSING', f)
    if not ok: all_ok = False
print()
print('All files present — ready!' if all_ok else 'FIX MISSING FILES before continuing!')


## Step 3 — HuggingFace Login + Download Svarah Dataset

**Svarah is a gated dataset.** You must:
1. Accept terms at [huggingface.co/datasets/ai4bharat/Svarah](https://huggingface.co/datasets/ai4bharat/Svarah)
2. Paste your HF token below (from huggingface.co → Settings → Access Tokens)

**Known Svarah quirks fixed in this notebook:**
- Split name is `test` (not `train`)
- Audio key is `audio_filepath` (not `audio`)


In [ ]:
from huggingface_hub import login

# ✏️ Paste your HuggingFace token here
HF_TOKEN = 'hf_XXXXXXXXXXXXXXXXXXXX'  # <-- replace with your token

login(token=HF_TOKEN)
print('HuggingFace login successful!')


In [ ]:
import os, numpy as np
from datasets import load_dataset
from tqdm import tqdm

os.chdir('/content/FastSpeech2')

print('Loading Svarah dataset (split=test)...')
# FIX 1: split is 'test' not 'train'
# FIX 2: trust_remote_code=True required
dataset = load_dataset(
    'ai4bharat/Svarah',
    split='test',
    trust_remote_code=True
)
print(f'Total samples: {len(dataset)}')
print(f'Features: {dataset.features}')

# Inspect the audio key name from features
sample0 = dataset[0]
print('Sample keys:', list(sample0.keys()))

# FIX 3: Svarah uses 'audio_filepath' not 'audio'
# Auto-detect the audio key
AUDIO_KEY = None
for candidate in ['audio', 'audio_filepath', 'speech', 'file']:
    if candidate in sample0:
        # Check if it's an Audio feature (has 'array')
        val = sample0[candidate]
        if isinstance(val, dict) and 'array' in val:
            AUDIO_KEY = candidate
            break
if AUDIO_KEY is None:
    raise KeyError(f'Cannot find audio key in sample. Keys: {list(sample0.keys())}')
print(f'Using audio key: {AUDIO_KEY!r}')

# Auto-detect text key
TEXT_KEY = None
for candidate in ['text', 'sentence', 'transcript', 'transcription', 'normalized_text']:
    if candidate in sample0:
        TEXT_KEY = candidate
        break
print(f'Using text key: {TEXT_KEY!r}')

# Select ~6 hours total (5h train + 1h val)
TARGET_TRAIN_SEC = 5.0 * 3600
TARGET_TOTAL_SEC = 6.0 * 3600

accumulated   = 0.0
selected      = []   # list of (idx, duration)

for i in tqdm(range(len(dataset)), desc='Scanning dataset'):
    sample = dataset[i]
    audio  = sample[AUDIO_KEY]
    dur    = len(audio['array']) / audio['sampling_rate']
    selected.append((i, dur))
    accumulated += dur
    if accumulated >= TARGET_TOTAL_SEC:
        break

print(f'Selected {len(selected)} samples = {accumulated/3600:.2f} hours')

# Split: first 5h = train, last 1h = val
train_indices, val_indices = [], []
cum = 0.0
for idx, dur in selected:
    if cum < TARGET_TRAIN_SEC:
        train_indices.append(idx)
    else:
        val_indices.append(idx)
    cum += dur
print(f'Train: {len(train_indices)} | Val: {len(val_indices)}')


In [ ]:
import csv, librosa, soundfile as sf
from tqdm import tqdm

metadata = []

for split_name, indices in [('train', train_indices), ('val', val_indices)]:
    for idx in tqdm(indices, desc=f'Saving {split_name}'):
        sample     = dataset[idx]
        audio_obj  = sample[AUDIO_KEY]
        audio_arr  = np.array(audio_obj['array'], dtype=np.float32)
        orig_sr    = audio_obj['sampling_rate']
        text       = str(sample.get(TEXT_KEY, '')) if TEXT_KEY else ''

        # Resample to 22050 Hz
        if orig_sr != 22050:
            audio_arr = librosa.resample(audio_arr, orig_sr=orig_sr, target_sr=22050)

        # FIX: proper f-string file_id
        file_id  = f'svarah_{idx:06d}'
        wav_path = f'/content/FastSpeech2/data/raw/english/wavs/{file_id}.wav'
        sf.write(wav_path, audio_arr, 22050)

        duration = len(audio_arr) / 22050
        metadata.append({
            'file_id':  file_id,
            'text':     text,
            'duration': round(duration, 4),
            'split':    split_name,
        })

# Save metadata.csv
meta_path = '/content/FastSpeech2/data/raw/english/metadata.csv'
with open(meta_path, 'w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['file_id', 'text', 'duration', 'split'])
    writer.writeheader()
    writer.writerows(metadata)

print(f'Saved {len(metadata)} audio files and metadata.csv')
print(f'Train: {sum(1 for m in metadata if m["split"]=="train")} | '
      f'Val: {sum(1 for m in metadata if m["split"]=="val")}')


## Step 4 — Preprocessing

1. **MFA corpus prep** — copy wavs + write `.lab` transcript files
2. **MFA alignment** — force-align phonemes to audio (~20–40 min)
3. **Feature extraction** — mel spectrograms, pitch (F0), energy


In [ ]:
import os, shutil, pandas as pd
from tqdm import tqdm

os.chdir('/content/FastSpeech2')

MFA_CORPUS = '/content/FastSpeech2/data/mfa_corpus/english'
MFA_OUT    = '/content/FastSpeech2/data/mfa_aligned/english'
os.makedirs(MFA_CORPUS, exist_ok=True)
os.makedirs(MFA_OUT,    exist_ok=True)

meta = pd.read_csv('/content/FastSpeech2/data/raw/english/metadata.csv')
print(f'Preparing MFA corpus for {len(meta)} files...')

skipped = 0
for _, row in tqdm(meta.iterrows(), total=len(meta)):
    fid  = row['file_id']
    text = str(row['text']).strip()
    if not text or text == 'nan':
        skipped += 1
        continue

    src_wav = f'/content/FastSpeech2/data/raw/english/wavs/{fid}.wav'
    if not os.path.exists(src_wav):
        skipped += 1
        continue

    shutil.copy2(src_wav, f'{MFA_CORPUS}/{fid}.wav')
    with open(f'{MFA_CORPUS}/{fid}.lab', 'w', encoding='utf-8') as fh:
        fh.write(text.lower())

print(f'MFA corpus ready. Skipped {skipped} files with empty/missing text.')
print(f'Total files in corpus: {len(os.listdir(MFA_CORPUS)) // 2}')


In [ ]:
import os

CONDA_DIR  = os.path.expanduser('~/miniconda')
MFA_BIN    = f'{CONDA_DIR}/bin/mfa'
MFA_CORPUS = '/content/FastSpeech2/data/mfa_corpus/english'
MFA_OUT    = '/content/FastSpeech2/data/mfa_aligned/english'

print('Running MFA alignment (20-40 minutes)...')
!{MFA_BIN} align \
    {MFA_CORPUS} \
    english_us_arpa \
    english_us_arpa \
    {MFA_OUT} \
    --clean \
    --num_jobs 4 \
    --quiet

# Count aligned files
tg_files = [f for f in os.listdir(MFA_OUT) if f.endswith('.TextGrid')]
print(f'MFA alignment complete! Aligned {len(tg_files)} files.')


In [ ]:
import os, sys
os.chdir('/content/FastSpeech2')
sys.path.insert(0, '/content/FastSpeech2')

print('Running preprocessing...')
# Use /usr/bin/python3 (Colab system Python with torch)
# NOT 'python' which might resolve to conda's python after Miniconda install
!/usr/bin/python3 English/preprocess.py --config English/config.yaml
print('Preprocessing complete!')


## Step 5 — Training

Expected time on **Colab T4**: ~8–12 hours for full training.
Checkpoints saved every 10,000 steps — Colab can resume from latest checkpoint.

> 💡 Training logs are streamed to Google Drive so you can monitor progress even if the tab goes idle.


In [ ]:
import os
os.chdir('/content/FastSpeech2')

# Create drive log dir
os.makedirs('/content/drive/MyDrive/FastSpeech2', exist_ok=True)

# Run training with system Python (torch guaranteed available)
!/usr/bin/python3 English/train.py --config English/config.yaml \
    2>&1 | tee /content/drive/MyDrive/FastSpeech2/english_training.log

print('Training complete!')


## Step 6 — Inference on Test Sentences

| ID | Sentence |
|----|----------|
| E1  | The antidisestablishmentarianism movement gained considerable momentum during the parliamentary proceedings. |
| E2a | The orchestra's conduct during the ceremony was praised by the audience. |
| E2b | The researchers will conduct an experiment to evaluate the proposed method. |
| E3  | The rhythm of the orchestra changed unexpectedly. |
| E4  | The meeting will start at 10:45 AM on 12 November 2026, so andaroo should arrive on time. |


In [ ]:
import os, subprocess
os.chdir('/content/FastSpeech2')

# --- Download HiFi-GAN repo + pretrained checkpoint ---
HIFIGAN_DIR = '/content/hifi-gan'

if not os.path.exists(HIFIGAN_DIR):
    !git clone https://github.com/jik876/hifi-gan {HIFIGAN_DIR}
    os.chdir(HIFIGAN_DIR)
    !pip install -q -r requirements.txt
    os.chdir('/content/FastSpeech2')

CKPT_PATH = f'{HIFIGAN_DIR}/generator_v1'
CFG_PATH  = f'{HIFIGAN_DIR}/config_v1.json'

if not os.path.exists(CKPT_PATH):
    print('Downloading HiFi-GAN pretrained checkpoint via gdown...')
    # UNIVERSAL_V1 (most robust — works for non-LJSpeech data)
    !gdown '1cbHgchar3GUjMlI2kK5XltHDxPMi-6b2' -O {CKPT_PATH}
    !wget -q https://raw.githubusercontent.com/jik876/hifi-gan/master/config_v1.json \
        -O {CFG_PATH}
    # rename checkpoint key to match generator_v1
    print('Checkpoint downloaded!')
else:
    print('HiFi-GAN checkpoint already present.')

# --- Run inference ---
os.chdir('/content/FastSpeech2')
CKPT = 'English/checkpoints/best_checkpoint.pth'
if not os.path.exists(CKPT):
    # Use latest checkpoint if 'best' not saved yet
    ckpts = sorted([f for f in os.listdir('English/checkpoints') if f.endswith('.pth')])
    CKPT  = f'English/checkpoints/{ckpts[-1]}' if ckpts else CKPT
    print(f'Using checkpoint: {CKPT}')

!/usr/bin/python3 English/inference.py \
    --checkpoint {CKPT} \
    --config     English/config.yaml \
    --output_dir Results/English \
    --vocoder_path {HIFIGAN_DIR}

print('Inference complete!')


In [ ]:
import os, glob
import IPython.display as ipd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

os.chdir('/content/FastSpeech2')

# Create dirs if somehow missing
os.makedirs('Results/English/test_generated_audios', exist_ok=True)
for t in ('duration', 'pitch', 'energy'):
    os.makedirs(f'Results/English/variance_adaptor_plots/{t}', exist_ok=True)

# --- Play audio ---
wav_files = sorted(glob.glob('Results/English/test_generated_audios/*.wav'))
if wav_files:
    print(f'=== {len(wav_files)} Generated Audio Samples ===')
    for w in wav_files:
        print(os.path.basename(w))
        display(ipd.Audio(w))
else:
    print('No audio files found — inference may not have completed yet.')

# --- Show variance adaptor plots ---
sent_ids = ['E1', 'E2a', 'E2b', 'E3', 'E4']
ptypes   = ['duration', 'pitch', 'energy']
fig, axes = plt.subplots(3, 5, figsize=(22, 10))

any_plot = False
for col, sid in enumerate(sent_ids):
    for row, pt in enumerate(ptypes):
        path = f'Results/English/variance_adaptor_plots/{pt}/{sid}_{pt}.png'
        ax   = axes[row][col]
        if os.path.exists(path):
            ax.imshow(mpimg.imread(path))
            any_plot = True
        else:
            ax.text(0.5, 0.5, 'Not generated', ha='center', va='center')
        ax.axis('off')
        if row == 0: ax.set_title(sid, fontsize=11)
        if col == 0: ax.set_ylabel(pt, fontsize=10)

plt.suptitle('Variance Adaptor Outputs — English', fontsize=14)
plt.tight_layout()
os.makedirs('Results/English', exist_ok=True)
plt.savefig('Results/English/all_variance_plots.png', dpi=100, bbox_inches='tight')
plt.show()
print('Plot saved to Results/English/all_variance_plots.png')


In [ ]:
import shutil, os
os.chdir('/content/FastSpeech2')

print('Saving results to Google Drive...')

# Results
if os.path.isdir('Results/English'):
    dst = '/content/drive/MyDrive/FastSpeech2/Results/English'
    os.makedirs(dst, exist_ok=True)
    shutil.copytree('Results/English', dst, dirs_exist_ok=True)
    print('Results saved to Drive.')

# Checkpoints
if os.path.isdir('English/checkpoints') and os.listdir('English/checkpoints'):
    dst = '/content/drive/MyDrive/FastSpeech2/English/checkpoints'
    os.makedirs(dst, exist_ok=True)
    shutil.copytree('English/checkpoints', dst, dirs_exist_ok=True)
    print('Checkpoints saved to Drive.')
else:
    print('No checkpoints found (training may not have finished yet).')

print()
print('All done! Check Google Drive at: MyDrive/FastSpeech2/')
